# Data Analysis SEW Motor

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# === 1. LOAD DATA ===
df = pd.read_csv("../data/sample_data.csv")
print(f"Original dataset shape: {df.shape} ({df.shape[0]} rows x {df.shape[1]} columns)")
print(f"Original memory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

# === 2. COLUMN CLEANING ===
# Remove constant columns (zero variance - useless for PCA)
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
print(f"\nConstant columns removed ({len(constant_cols)}): {constant_cols}")

# Keep only numeric, non-constant columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in constant_cols]

# Remove raw columns that have an engineering-unit equivalent (avoid duplicates)
raw_to_remove = []
for col in numeric_cols:
    if not any(u in col for u in ['_rpm', '_%', '_degC', '_V', '_A', '_Hz', '_scaled']):
        scaled_versions = [c for c in numeric_cols if c.startswith(col + '_') and c != col]
        if scaled_versions:
            raw_to_remove.append(col)

print(f"Raw duplicate columns removed ({len(raw_to_remove)}): {raw_to_remove}")

# Final feature set
pca_cols = [c for c in numeric_cols if c not in raw_to_remove]
print(f"\nFinal features for PCA ({len(pca_cols)}):")
for i, c in enumerate(pca_cols, 1):
    print(f"  {i:>2}. {c}")

# === 3. PREPARE & STANDARDIZE ===
X = df[pca_cols]
feature_names = pca_cols
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# === 4. PCA ===
pca = PCA()
scores = pca.fit_transform(X_scaled)

# === 5. EXPLAINED VARIANCE ===
print("\n" + "="*60)
print("EXPLAINED VARIANCE PER COMPONENT")
print("="*60)
print(f"{'PC':<5} {'Variance %':<15} {'Cumulative %':<15}")
print("-"*35)
for i, (var, cum) in enumerate(zip(pca.explained_variance_ratio_, 
                                    np.cumsum(pca.explained_variance_ratio_))):
    print(f"PC{i+1:<3} {var*100:>8.2f}%       {cum*100:>8.2f}%")

n_comp_90 = np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.90) + 1
n_comp_95 = np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.95) + 1
print(f"\nComponents needed for 90% variance: {n_comp_90}")
print(f"Components needed for 95% variance: {n_comp_95}")

# === 6. DOMINANT FEATURE PER COMPONENT ===
# For each PC, which REAL feature has the highest loading?
loadings = pca.components_.T  # shape: (n_features, n_components)

print("\n" + "="*60)
print("DOMINANT FEATURE FOR EACH PRINCIPAL COMPONENT")
print("(The single real variable that best represents each PC)")
print("="*60)
for i in range(n_comp_95):
    abs_loadings = np.abs(loadings[:, i])
    top3_idx = np.argsort(abs_loadings)[::-1][:3]
    var_pct = pca.explained_variance_ratio_[i] * 100
    print(f"\n  PC{i+1} ({var_pct:.1f}% variance):")
    print(f"    #1: {feature_names[top3_idx[0]]:<30} loading = {loadings[top3_idx[0], i]:+.3f}")
    print(f"    #2: {feature_names[top3_idx[1]]:<30} loading = {loadings[top3_idx[1], i]:+.3f}")
    print(f"    #3: {feature_names[top3_idx[2]]:<30} loading = {loadings[top3_idx[2], i]:+.3f}")

# === 7. CORRELATION CLUSTERING ===
# Group features that are highly correlated (|r| > 0.9) — you only need ONE per group
print("\n" + "="*60)
print("CORRELATION CLUSTERS (|r| > 0.9)")
print("Features in the same group are redundant — keep only one per group")
print("="*60)

corr_matrix = np.corrcoef(X_scaled.T)

#Set a threshold (absolute Pearson coefficient) for correlation-based variable reduction

threshold = 0.9
visited = set()
clusters = []

for i in range(len(feature_names)):
    if i in visited:
        continue
    cluster = [i]
    visited.add(i)
    for j in range(i+1, len(feature_names)):
        if j not in visited and abs(corr_matrix[i, j]) > threshold:
            cluster.append(j)
            visited.add(j)
    clusters.append(cluster)

print(f"\nFound {len(clusters)} independent groups:\n")
representative_features = []
for idx, cluster in enumerate(clusters, 1):
    names = [feature_names[i] for i in cluster]
    # Pick the one with highest overall variance contribution as representative
    variances = [np.var(X_scaled[:, i]) for i in cluster]
    best_idx = cluster[np.argmax(variances)]
    representative_features.append(feature_names[best_idx])
    
    if len(names) > 1:
        print(f"  Group {idx} ({len(names)} features, highly correlated):")
        for n in names:
            marker = " <-- KEEP THIS ONE" if n == feature_names[best_idx] else ""
            print(f"    - {n}{marker}")
    else:
        print(f"  Group {idx} (independent):")
        print(f"    - {names[0]}")

print(f"\n{'='*60}")
print(f"RECOMMENDED MINIMAL FEATURE SET ({len(representative_features)} features)")
print(f"One representative per correlation group:")
print(f"{'='*60}")
for i, f in enumerate(representative_features, 1):
    print(f"  {i:>2}. {f}")

# === 8. FEATURE IMPORTANCE RANKING ===
# Importance = sum of squared loadings weighted by explained variance
feature_importance = np.sum(
    (loadings[:, :n_comp_95] ** 2) * pca.explained_variance_ratio_[:n_comp_95], 
    axis=1
)
feature_importance_pct = (feature_importance / feature_importance.sum()) * 100

ranking = pd.DataFrame({
    'Feature': feature_names,
    'Importance_%': feature_importance_pct
}).sort_values('Importance_%', ascending=False).reset_index(drop=True)
ranking.index += 1

print("\n" + "="*60)
print("FEATURE IMPORTANCE RANKING (all features)")
print("="*60)
print(ranking.to_string())

# === 9. TOP 10 FEATURES - STATISTICS ===
print("\n" + "="*60)
print("TOP 10 MOST IMPORTANT FEATURES - STATISTICS")
print("="*60)
top10 = ranking.head(10)['Feature'].tolist()
for feat in top10:
    col_data = df[feat]
    print(f"\n  {feat}:")
    print(f"    Mean:  {col_data.mean():.4f}")
    print(f"    Std:   {col_data.std():.4f}")
    print(f"    Min:   {col_data.min():.4f}")
    print(f"    Max:   {col_data.max():.4f}")
    print(f"    Range: {col_data.max() - col_data.min():.4f}")

# === 10. MEMORY & DIMENSIONALITY REDUCTION ===
print("\n" + "="*60)
print("MEMORY & DIMENSIONALITY REDUCTION")
print("="*60)

mem_original = df.select_dtypes(include=[np.number]).memory_usage(deep=True).sum()
mem_cleaned = df[pca_cols].memory_usage(deep=True).sum()
mem_pca_95 = scores[:, :n_comp_95].nbytes
mem_pca_90 = scores[:, :n_comp_90].nbytes
mem_minimal = df[representative_features].memory_usage(deep=True).sum()

print(f"\n  COLUMNS:")
print(f"    Original total columns:           {df.shape[1]}")
print(f"    Original numeric columns:         {len(df.select_dtypes(include=[np.number]).columns)}")
print(f"    After cleaning (PCA input):       {len(pca_cols)}")
print(f"    After PCA (95% variance):         {n_comp_95} abstract components")
print(f"    After PCA (90% variance):         {n_comp_90} abstract components")
print(f"    Minimal real features (clusters): {len(representative_features)} real columns")

print(f"\n  MEMORY:")
print(f"    Original numeric data:            {mem_original / 1024:.2f} KB")
print(f"    After column cleaning:            {mem_cleaned / 1024:.2f} KB")
print(f"    After PCA (95% variance):         {mem_pca_95 / 1024:.2f} KB")
print(f"    After PCA (90% variance):         {mem_pca_90 / 1024:.2f} KB")
print(f"    Minimal real features:            {mem_minimal / 1024:.2f} KB")

print(f"\n  REDUCTION SUMMARY:")
print(f"    {df.shape[1]} original cols -> {len(representative_features)} real features ({(1 - len(representative_features)/df.shape[1])*100:.0f}% column reduction)")
print(f"    {mem_original/1024:.1f} KB -> {mem_minimal/1024:.1f} KB ({(1 - mem_minimal/mem_original)*100:.0f}% memory saved)")

# === 11. PLOTS ===
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Scree plot
ax = axes[0, 0]
ax.bar(range(1, len(pca.explained_variance_ratio_)+1), 
       pca.explained_variance_ratio_*100, alpha=0.7, color='steelblue', label='Individual')
ax.plot(range(1, len(pca.explained_variance_ratio_)+1), 
        np.cumsum(pca.explained_variance_ratio_)*100, 'ro-', markersize=5, label='Cumulative')
ax.axhline(y=95, color='g', linestyle='--', alpha=0.7, label='95% threshold')
ax.axhline(y=90, color='orange', linestyle='--', alpha=0.7, label='90% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance (%)')
ax.set_title('Scree Plot - Explained Variance')
ax.legend()

# Feature importance
ax = axes[0, 1]
top_n = 15
top = ranking.head(top_n)
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, top_n))
ax.barh(range(top_n-1, -1, -1), top['Importance_%'].values, color=colors)
ax.set_yticks(range(top_n-1, -1, -1))
ax.set_yticklabels(top['Feature'].values, fontsize=8)
ax.set_xlabel('Importance (%)')
ax.set_title('Top 15 Features by PCA Importance')
ax.axvline(x=100/len(pca_cols), color='gray', linestyle=':', alpha=0.7, label='Average (equal importance)')
ax.legend()

# Correlation heatmap (to see the clusters)
ax = axes[1, 0]
im = ax.imshow(np.abs(corr_matrix), cmap='RdYlBu_r', vmin=0, vmax=1)
ax.set_xticks(range(len(feature_names)))
ax.set_yticks(range(len(feature_names)))
ax.set_xticklabels([f.split('.')[-1] for f in feature_names], rotation=90, fontsize=6)
ax.set_yticklabels([f.split('.')[-1] for f in feature_names], fontsize=6)
ax.set_title('Correlation Matrix (absolute values)\nBright blocks = redundant features')
plt.colorbar(im, ax=ax)

# Data projection
ax = axes[1, 1]
scatter = ax.scatter(scores[:, 0], scores[:, 1], c=range(len(scores)), 
                     cmap='viridis', alpha=0.6, s=10)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Data Projection - PC1 vs PC2 (color = time)')
plt.colorbar(scatter, ax=ax, label='Sample index (time)')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)

plt.tight_layout()
# plt.savefig('pca_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# === FINAL SUMMARY: 9 REAL FEATURES ===
print("\n" + "="*60)
print(f"YOUR MINIMAL DATASET: {len(representative_features)} FEATURES")
print("These are the only columns you need for analysis:")
print("="*60)

df_minimal = df[representative_features].copy()

for i, feat in enumerate(representative_features, 1):
    col = df_minimal[feat]
    print(f"\n  {i:>2}. {feat}")
    print(f"      Mean: {col.mean():.4f}  |  Std: {col.std():.4f}  |  Min: {col.min():.4f}  |  Max: {col.max():.4f}")

print(f"\n{'='*60}")
print("REDUCTION SUMMARY")
print("="*60)
mem_original = df.memory_usage(deep=True).sum()
mem_minimal = df_minimal.memory_usage(deep=True).sum()
print(f"  Original:  {df.shape[1]} columns, {mem_original/1024:.1f} KB")
print(f"  Reduced:   {len(representative_features)} columns, {mem_minimal/1024:.1f} KB")
print(f"  Savings:   {(1 - len(representative_features)/df.shape[1])*100:.0f}% fewer columns, {(1 - mem_minimal/mem_original)*100:.0f}% less memory")
print(f"\n  Columns to keep:")
print(f"  {representative_features}")

In [ ]:
# === PC1 & PC2 COMPOSITION ===
print("\n" + "="*60)
print("PC1 COMPOSITION (what's inside PC1)")
print(f"Explained variance: {pca.explained_variance_ratio_[0]*100:.1f}%")
print("="*60)

# Sort by absolute loading
pc1_loadings = pd.DataFrame({
    'Feature': feature_names,
    'Loading': loadings[:, 0]
}).sort_values('Loading', key=abs, ascending=False)

for _, row in pc1_loadings.iterrows():
    bar = "+" * int(abs(row['Loading']) * 30) if row['Loading'] > 0 else "-" * int(abs(row['Loading']) * 30)
    print(f"  {row['Feature']:<30} {row['Loading']:+.4f}  {bar}")

print("\n" + "="*60)
print("PC2 COMPOSITION (what's inside PC2)")
print(f"Explained variance: {pca.explained_variance_ratio_[1]*100:.1f}%")
print("="*60)

pc2_loadings = pd.DataFrame({
    'Feature': feature_names,
    'Loading': loadings[:, 1]
}).sort_values('Loading', key=abs, ascending=False)

for _, row in pc2_loadings.iterrows():
    bar = "+" * int(abs(row['Loading']) * 30) if row['Loading'] > 0 else "-" * int(abs(row['Loading']) * 30)
    print(f"  {row['Feature']:<30} {row['Loading']:+.4f}  {bar}")

# === HOW MUCH VARIANCE DO THE 9 REPRESENTATIVE FEATURES EXPLAIN? ===
print("\n" + "="*60)
print(f"VARIANCE RETAINED BY THE {len(representative_features)} SELECTED FEATURES")
print("="*60)

# Run PCA only on the representative features
X_minimal = df[representative_features].values.astype(float)
X_minimal_scaled = (X_minimal - X_minimal.mean(axis=0)) / X_minimal.std(axis=0)

# Variance of minimal set compared to full set
# We project the full scaled data onto the subspace of representative features
# and measure how much of the total variance is captured
from sklearn.decomposition import PCA as PCA2

pca_full = PCA2()
pca_full.fit(X_scaled)  # full 25 features

# Indices of representative features in the full feature list
rep_indices = [feature_names.index(f) for f in representative_features]

# Total variance in full dataset (after standardization = number of features)
total_variance_full = len(feature_names)  # sum of all eigenvalues = n_features for standardized data

# Variance explained by keeping only the representative features
# = sum of variances of those columns in the original standardized space
variance_kept = sum(np.var(X_scaled[:, i], ddof=0) for i in rep_indices)
# Since standardized, each column has var=1, so variance_kept ~ len(rep_indices)
# But the REAL question is: how much of the OTHER columns' info do these 9 capture?

# Better approach: R² of predicting all 25 features from the 9 representatives
from numpy.linalg import lstsq

X_rep = X_scaled[:, rep_indices]
total_var = np.sum(np.var(X_scaled, axis=0))
explained_var = 0

print(f"\n  Total features: {len(feature_names)}")
print(f"  Selected features: {len(representative_features)}")
print(f"\n  Variance of each selected feature (should be ~1.0 if standardized):")
for f in representative_features:
    idx = feature_names.index(f)
    print(f"    {f:<30} var = {np.var(X_scaled[:, idx]):.4f}")

# R² : how well do the 9 features reconstruct ALL 25?
residual_var = 0
for j in range(X_scaled.shape[1]):
    coeffs, _, _, _ = lstsq(X_rep, X_scaled[:, j], rcond=None)
    predictions = X_rep @ coeffs
    residual_var += np.var(X_scaled[:, j] - predictions)

r_squared = 1 - (residual_var / total_var)

print(f"\n  R² (how well 9 features reconstruct all 25): {r_squared*100:.1f}%")
print(f"  This means your 9 features retain {r_squared*100:.1f}% of the total information.")
print(f"\n  Comparison:")
print(f"    PCA with {n_comp_95} components: 95.0% variance (abstract, not real columns)")
print(f"    Your {len(representative_features)} real features: {r_squared*100:.1f}% variance (actual columns you can interpret)")

## Remark

The remaining 3.9% of variance lost accounts for rare decoupling between sensors. The correlation structure should hold under normal operating conditions. If rare edge cases break it, extending from 9 to 25 features may be warranted — but for initial testing, 9 features are sufficient. The `threshold` variable can be tuned: lower it to keep more features and increase precision.

## Reduction Summary (from production data run)

> **Note**: The output below is from the original analysis on production telemetry data.
> Running with synthetic data will produce different specific features, but the methodology
> and reduction ratios are representative.

```
============================================================
REDUCTION SUMMARY
============================================================
  Original:  52 columns, 311.8 KB
  Reduced:   9 columns, 42.3 KB
  Savings:   83% fewer columns, 86% less memory

  Columns to keep:
  ['PositionUser', 'ActiveCurrent_A', 'MotorTemp_C', 'RelExcitCurrent',
   'ExcitCurrent_A', 'DcBusVoltage_V', 'ElecMechUtil_%', 'HeatsinkUtil_%', 'TotalUtil_%']
```


In [ ]:
# === FEATURE SELECTION (independent from PCA/clustering) ===
# Uses multiple methods to rank features by importance
# No target variable needed — unsupervised selection

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler

# === 1. LOAD DATA ===
df = pd.read_csv("../data/sample_data.csv")

# Same cleaning as before: remove constants, timestamp, raw duplicates
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in constant_cols]

raw_to_remove = []
for col in numeric_cols:
    if not any(u in col for u in ['_rpm', '_%', '_degC', '_V', '_A', '_Hz', '_scaled']):
        scaled_versions = [c for c in numeric_cols if c.startswith(col + '_') and c != col]
        if scaled_versions:
            raw_to_remove.append(col)

pca_cols = [c for c in numeric_cols if c not in raw_to_remove]
X = df[pca_cols].values.astype(float)
feature_names = pca_cols

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("="*60)
print("FEATURE SELECTION (independent from PCA/clustering)")
print("="*60)

# === METHOD 1: Variance ranking ===
# Features with more variance carry more information
variances = np.var(X, axis=0)  # on raw data, not scaled
var_ranking = np.argsort(variances)[::-1]

print("\n--- METHOD 1: Variance (raw scale) ---")
print("Features with most variation in their values:")
for i, idx in enumerate(var_ranking[:10], 1):
    print(f"  {i:>2}. {feature_names[idx]:<30} variance = {variances[idx]:.2f}")

# === METHOD 2: Mutual Information (each feature vs all others combined) ===
# How much info does each feature carry about the rest of the dataset?
print("\n--- METHOD 2: Mutual Information (unsupervised) ---")
print("How much unique information each feature carries about the others:")

mi_scores = np.zeros(len(feature_names))
for i in range(len(feature_names)):
    # Target = this feature, inputs = all others
    others = np.delete(X_scaled, i, axis=1)
    mi = mutual_info_regression(others, X_scaled[:, i], random_state=42, n_neighbors=5)
    mi_scores[i] = mi.mean()

# Invert: LOW mutual info = feature is UNIQUE (hard to predict from others)
uniqueness = 1.0 / (mi_scores + 1e-10)
uniqueness_pct = (uniqueness / uniqueness.sum()) * 100
mi_ranking = np.argsort(uniqueness_pct)[::-1]

for i, idx in enumerate(mi_ranking[:10], 1):
    print(f"  {i:>2}. {feature_names[idx]:<30} uniqueness = {uniqueness_pct[idx]:.2f}%")

# === METHOD 3: Random Forest (reconstruct each feature from others) ===
# Features that are HARD to predict from the rest are the most unique/important
print("\n--- METHOD 3: Random Forest reconstruction error ---")
print("Features hardest to reconstruct from others (most unique info):")

rf_errors = np.zeros(len(feature_names))
for i in range(len(feature_names)):
    others = np.delete(X_scaled, i, axis=1)
    target = X_scaled[:, i]
    rf = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
    rf.fit(others, target)
    predictions = rf.predict(others)
    # Higher error = harder to predict = more unique
    rf_errors[i] = np.mean((target - predictions) ** 2)

rf_error_pct = (rf_errors / rf_errors.sum()) * 100
rf_ranking = np.argsort(rf_error_pct)[::-1]

for i, idx in enumerate(rf_ranking[:10], 1):
    print(f"  {i:>2}. {feature_names[idx]:<30} reconstruction error = {rf_error_pct[idx]:.2f}%")

# === METHOD 4: Laplacian Score (preserves local structure) ===
# Features that best preserve the neighborhood structure of the data
#print("\n--- METHOD 4: Laplacian Score ---")
#print("Features that best preserve data structure (local neighborhoods):")

from scipy.spatial.distance import pdist, squareform

# Build k-nearest neighbor graph
#distances = squareform(pdist(X_scaled, 'euclidean'))
#k = 10
#W = np.zeros_like(distances)
#for i in range(len(X_scaled)):
    #neighbors = np.argsort(distances[i])[:k+1]
    #for j in neighbors:
        #W[i, j] = np.exp(-distances[i, j]**2 / (2 * np.median(distances)**2))
        #W[j, i] = W[i, j]

#D = np.diag(W.sum(axis=1))
#L = D - W

#laplacian_scores = np.zeros(len(feature_names))
#for i in range(len(feature_names)):
    #f = X_scaled[:, i]
    #f_tilde = f - (f @ D @ np.ones(len(f))) / (np.ones(len(f)) @ D @ np.ones(len(f)))
    #laplacian_scores[i] = (f_tilde @ L @ f_tilde) / (f_tilde @ D @ f_tilde + 1e-10)

# Lower score = better (preserves structure more)
#lap_ranking = np.argsort(laplacian_scores)

#for i, idx in enumerate(lap_ranking[:10], 1):
    #print(f"  {i:>2}. {feature_names[idx]:<30} laplacian score = {laplacian_scores[idx]:.4f}")

# === COMBINED RANKING (consensus across all methods) ===
print("\n" + "="*60)
print("COMBINED RANKING (average rank across all 4 methods)")
print("="*60)

# Build rank per method (1 = best)
ranks = np.zeros((len(feature_names), 3))
ranks[var_ranking, 0] = np.arange(len(feature_names))
ranks[mi_ranking, 1] = np.arange(len(feature_names))
ranks[rf_ranking, 2] = np.arange(len(feature_names))
#ranks[lap_ranking, 3] = np.arange(len(feature_names))

avg_rank = ranks.mean(axis=1)
combined_ranking = np.argsort(avg_rank)

#print(f"\n{'#':<4} {'Feature':<30} {'Avg Rank':<10} {'Var':<6} {'MI':<6} {'RF':<6} {'Lap':<6}")
#print("-"*68)
#for i, idx in enumerate(combined_ranking, 1):
    #print(f"{i:<4} {feature_names[idx]:<30} {avg_rank[idx]:<10.1f} {int(ranks[idx,0]+1):<6} {int(ranks[idx,1]+1):<6} {int(ranks[idx,2]+1):<6} {int(ranks[idx,3]+1):<6}")

# === TOP 9 FEATURES (to compare with clustering result) ===
top9_selection = [feature_names[idx] for idx in combined_ranking[:9]]
print(f"\n{'='*60}")
print(f"FEATURE SELECTION: TOP 9 RECOMMENDED FEATURES")
print(f"{'='*60}")
for i, f in enumerate(top9_selection, 1):
    print(f"  {i:>2}. {f}")

# === COMPARISON WITH CLUSTERING ===
# Paste your 9 features from clustering here:
# clustering_features = ["feat1", "feat2", ...]  # <-- replace with your 9
# overlap = set(top9_selection) & set(clustering_features)
# print(f"\nOverlap with clustering: {len(overlap)}/{9} features in common")
# print(f"Common: {overlap}")
# print(f"Only in selection: {set(top9_selection) - set(clustering_features)}")
# print(f"Only in clustering: {set(clustering_features) - set(top9_selection)}")

In [ ]:
# === R² FOR SET B (Feature Selection results from production data) ===
# NOTE: These features were identified by the feature selection methods
# on production data. With synthetic data, different features will be selected.
# The dynamic cells above (1-6) will work with any data.

selection_features = [
    'TorqueRelMin', 'TorqueRelMax', 'TorqueRelActual',
    'RelExcitCurrent', 'VoltageRef_V', 'SpeedSys',
    'DcBusVoltage_V', 'RelActiveCurrent', 'OutputFreq_Hz'
]

# Guard: skip if columns don't exist (e.g., synthetic data)
missing = [f for f in selection_features if f not in feature_names]
if missing:
    print(f"⚠️  Skipping Set B analysis — {len(missing)} features not found in data: {missing}")
    print("   This is expected when running with synthetic data.")
    print("   The dynamic analysis cells above are sufficient.")
else:
    rep_indices_B = [feature_names.index(f) for f in selection_features]
    X_rep_B = X_scaled[:, rep_indices_B]
    r2_scores_B = []
    for col_idx in range(X_scaled.shape[1]):
        if col_idx in rep_indices_B:
            continue
        y = X_scaled[:, col_idx]
        coeffs, _, _, _ = np.linalg.lstsq(X_rep_B, y, rcond=None)
        y_pred = X_rep_B @ coeffs
        ss_res = np.sum((y - y_pred)**2)
        ss_tot = np.sum((y - y.mean())**2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 1.0
        r2_scores_B.append(r2 * 100)

    print(f"\n{'='*60}")
    print(f"SET B (Feature Selection): R² of {len(selection_features)} features")
    print(f"{'='*60}")
    print(f"  Mean R²: {np.mean(r2_scores_B):.1f}%")
    print(f"  Min R²:  {np.min(r2_scores_B):.1f}%")
    print(f"  Max R²:  {np.max(r2_scores_B):.1f}%")


In [ ]:
# === MERGE SET A (Clustering) + SET B (Feature Selection) ===
# NOTE: These hardcoded lists are from production data analysis.
# With synthetic data, the union will differ.

clustering_features = [
    'PositionUser', 'ActiveCurrent_A', 'MotorTemp_C', 'RelExcitCurrent',
    'ExcitCurrent_A', 'DcBusVoltage_V', 'ElecMechUtil_%', 'HeatsinkUtil_%', 'TotalUtil_%'
]

selection_features_B = [
    'TorqueRelMin', 'TorqueRelMax', 'TorqueRelActual',
    'RelExcitCurrent', 'VoltageRef_V', 'SpeedSys',
    'DcBusVoltage_V', 'RelActiveCurrent', 'OutputFreq_Hz'
]

# Guard: only run if all features exist in the data
all_needed = set(clustering_features + selection_features_B)
missing = [f for f in all_needed if f not in feature_names]

if missing:
    print(f"⚠️  Skipping Set A+B comparison — {len(missing)} features not found in data.")
    print(f"   Missing: {missing}")
    print(f"   This is expected with synthetic data. See the dynamic cells above.")
else:
    union_features = sorted(set(clustering_features + selection_features_B))
    only_A = sorted(set(clustering_features) - set(selection_features_B))
    only_B = sorted(set(selection_features_B) - set(clustering_features))
    both = sorted(set(clustering_features) & set(selection_features_B))

    print(f"{'='*60}")
    print(f"SET A (Clustering):        {len(clustering_features)} features")
    print(f"SET B (Feature Selection): {len(selection_features_B)} features")
    print(f"UNION (A ∪ B):             {len(union_features)} features")
    print(f"{'='*60}")
    print(f"\n  In both:  {both}")
    print(f"  Only A:   {only_A}")
    print(f"  Only B:   {only_B}")

    # R² for union
    union_indices = [feature_names.index(f) for f in union_features]
    X_union = X_scaled[:, union_indices]
    r2_union = []
    for col_idx in range(X_scaled.shape[1]):
        if col_idx in union_indices:
            continue
        y = X_scaled[:, col_idx]
        coeffs, _, _, _ = np.linalg.lstsq(X_union, y, rcond=None)
        y_pred = X_union @ coeffs
        ss_res = np.sum((y - y_pred)**2)
        ss_tot = np.sum((y - y.mean())**2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 1.0
        r2_union.append(r2 * 100)

    print(f"\n  UNION R²: {np.mean(r2_union):.1f}% mean")
    print(f"  → With {len(union_features)} features you can reconstruct")
    print(f"    {np.mean(r2_union):.1f}% of the removed features (on average)")


## Feature Selection Results: Two Complementary Sets

### Set A — Correlation Clustering (9 features)
**Answers the question: "What is the overall STATE of the drive?"**

Covers all physical domains:
- **Position**: `PositionUser`
- **Torque/Current**: `ActiveCurrent_A`, `RelExcitCurrent`, `ExcitCurrent_A`
- **Thermal**: `MotorTemp_C`, `HeatsinkUtil_%`
- **Electrical**: `DcBusVoltage_V`, `ElecMechUtil_%`, `TotalUtil_%`

Best for: general monitoring, anomaly detection, thermal degradation, long-term health tracking.

---

### Set B — Feature Selection (9 features)
**Answers the question: "What is the drive DOING right now?"**

Focuses on dynamic/instantaneous behavior:
- **Torque**: `TorqueRelMin`, `TorqueRelMax`, `TorqueRelActual`, `RelActiveCurrent`
- **Speed/Frequency**: `SpeedSys`, `OutputFreq_Hz`
- **Electrical**: `RelExcitCurrent`, `DcBusVoltage_V`, `VoltageRef_V`

Best for: real-time control analysis, load cycle characterization, dynamic fault detection.

---

### Summary
| | Set A (Clustering) | Set B (Feature Selection) |
|---|---|---|
| Focus | Full state coverage | Dynamic behavior |
| Strength | Catches slow drifts, thermal issues | Catches transient faults, load spikes |
| Weakness | Less detail on fast dynamics | Misses thermal/position info |
| R² vs full dataset | 96.1% | 60.9% |